# 03 — Linear Regression as a classification comparator

**Objective.** Examine whether ordinary least-squares regression provides useful scores for a binary machine-failure label, and identify the limitations of treating continuous regression outputs as failure alarms.

**Workflow.** Reconstruct the same development split → encode and scale inputs inside a pipeline → compute five-fold out-of-fold (OOF) predictions → inspect regression error, score range and classification results at 0.50.

Ordinary Linear Regression minimises squared prediction error; it is **not** Logistic Regression, and its outputs are not constrained to the interval [0, 1].

## 1. Reuse the approved features and row split

No identifiers or failure-mode indicators are used. The following cells explicitly recreate Notebook 02's 80/20 split from the same CSV using `random_state=42` and `stratify=y`.

In [13]:
from IPython.display import display
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from pathlib import Path
from sklearn.model_selection import train_test_split

# Exact data schema used throughout the original experiments.
RANDOM_STATE = 42
TARGET = "Machine failure"
CATEGORICAL_FEATURES = ["Type"]
NUMERIC_FEATURES = [
    "Air temperature [K]", "Process temperature [K]",
    "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]",
]
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

# Accept execution from the repository root or its notebooks/ directory.
# No extra .py helper file is needed.
DATA_PATH = Path("data/raw/ai4i2020.csv")
if not DATA_PATH.is_file():
    DATA_PATH = Path("../data/raw/ai4i2020.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "Cannot locate data/raw/ai4i2020.csv. Open the project root "
        "or its notebooks/ folder in VS Code and keep the original CSV in data/raw/. "
        f"Current working directory: {Path.cwd()}"
    )

df = pd.read_csv(DATA_PATH)
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")


Dataset loaded: 10,000 rows × 14 columns


In [14]:
# These six inputs are available at prediction time; failure-mode flags and IDs
# are deliberately excluded because they could leak the target or identify rows.
required_columns = FEATURES + [TARGET]
missing_columns = [name for name in required_columns if name not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")
if df[required_columns].isna().any().any():
    raise ValueError("Missing values found in the selected modelling columns.")

X = df[FEATURES].copy()
y = df[TARGET].copy()

# Use precisely the same stratified, seeded 80/20 row split in every notebook.
# Stratification maintains approximately 3.39% failures in each subset.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y,
)

print(f"Development/training set: {len(X_train):,}")
print(f"Previously inspected held-out test set: {len(X_test):,}")
print(f"Training failure rate: {100 * y_train.mean():.2f}%")
print(f"Test failure rate: {100 * y_test.mean():.2f}%")

Development/training set: 8,000
Previously inspected held-out test set: 2,000
Training failure rate: 3.39%
Test failure rate: 3.40%


## 2. Model and preprocessing

`StandardScaler` standardises the five numerical columns; `OneHotEncoder` converts `Type` into numerical indicator columns. Both steps live in a `Pipeline` so cross-validation fits them on each training fold only.

`StratifiedKFold` maintains the approximate failure ratio across five folds.

In [15]:
# ---------------------------------------------------------
# Preprocessing
# ---------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            NUMERIC_FEATURES,
        ),
        (
            "categorical",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
            ),
            CATEGORICAL_FEATURES,
        ),
    ]
)


# ---------------------------------------------------------
# Linear Regression model
# ---------------------------------------------------------

linear_model = LinearRegression()


# ---------------------------------------------------------
# Complete modelling pipeline
# ---------------------------------------------------------

linear_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", linear_model),
    ]
)


# ---------------------------------------------------------
# Stratified cross-validation
# ---------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print("Linear Regression pipeline created.")
print("Cross-validation: 5-fold stratified")

Linear Regression pipeline created.
Cross-validation: 5-fold stratified


## 3. Out-of-fold regression and classification results

`cross_val_predict()` generates one development-set prediction per observation from a model trained without that observation. MAE, RMSE and R² describe numeric error when fitting a 0/1 label. Applying an exploratory threshold of 0.50 converts scores into alarm/no-alarm decisions, **but does not turn them into calibrated failure probabilities**.

The score range and confusion matrix should be interpreted alongside the aggregate regression error: small numeric errors in a heavily imbalanced target do not guarantee detection of the minority class.

In [16]:
# ---------------------------------------------------------
# Generate out-of-fold predictions
# ---------------------------------------------------------
# Every prediction is produced by a model that did not
# train on that particular observation.

failure_score_oof = cross_val_predict(
    linear_pipeline,
    X_train,
    y_train,
    cv=cv,
    method="predict",
)


# ---------------------------------------------------------
# Regression performance
# ---------------------------------------------------------

regression_metrics = pd.DataFrame({
    "Score": {
        "MAE": mean_absolute_error(
            y_train,
            failure_score_oof,
        ),
        "RMSE": root_mean_squared_error(
            y_train,
            failure_score_oof,
        ),
        "R²": r2_score(
            y_train,
            failure_score_oof,
        ),
    }
})

print("Regression performance:")
display(regression_metrics.round(4))


# ---------------------------------------------------------
# Examine failure-score range
# ---------------------------------------------------------

minimum_score = failure_score_oof.min()
maximum_score = failure_score_oof.max()

below_zero = np.sum(
    failure_score_oof < 0
)

above_one = np.sum(
    failure_score_oof > 1
)

print(
    f"\nPredicted failure-score range: "
    f"{minimum_score:.4f} to "
    f"{maximum_score:.4f}"
)

print(
    f"Scores below 0: {below_zero:,}"
)

print(
    f"Scores above 1: {above_one:,}"
)


# ---------------------------------------------------------
# Convert failure score to class prediction
# ---------------------------------------------------------

CLASSIFICATION_THRESHOLD = 0.5

y_pred_oof = (
    failure_score_oof
    >= CLASSIFICATION_THRESHOLD
).astype(int)


# ---------------------------------------------------------
# Classification check (comparison only; not a probability model)
# ---------------------------------------------------------
print("\nClassification performance at score threshold 0.50:")
classification_scores = pd.DataFrame({"Score": {
    "Accuracy": accuracy_score(y_train, y_pred_oof),
    "Balanced Accuracy": balanced_accuracy_score(y_train, y_pred_oof),
    "Precision": precision_score(y_train, y_pred_oof, zero_division=0),
    "Recall": recall_score(y_train, y_pred_oof, zero_division=0),
    "F1-score": f1_score(y_train, y_pred_oof, zero_division=0),
}})
display(classification_scores.round(4))
print("\nCross-validated confusion matrix:")
cm = confusion_matrix(y_train, y_pred_oof, labels=[0, 1])
display(pd.DataFrame(cm,
    index=["Actual normal", "Actual failure"],
    columns=["Predicted normal", "Predicted failure"],
))

Regression performance:


,Score
MAE,0.0774
RMSE,0.1692
R²,0.1253



Predicted failure-score range: -0.1397 to 0.3526
Scores below 0: 2,571
Scores above 1: 0

Classification performance at score threshold 0.50:


,Score
Accuracy,0.9661
Balanced Accuracy,0.5000
Precision,0.0000
Recall,0.0000
F1-score,0.0000



Cross-validated confusion matrix:


,Predicted normal,Predicted failure
Actual normal,7729,0
Actual failure,271,0


## Linear Regression Findings and Limitations

Linear Regression was tested as an initial numerical benchmark using five-fold cross-validation.

It achieved MAE = 0.0774, RMSE = 0.1692 and R² = 0.1253. However, its predicted scores ranged from -0.1397 to 0.3526. At a threshold of 0.50, it detected none of the 271 development failures.

This shows that relatively small numerical prediction errors do not guarantee useful classification performance. Linear Regression can also produce negative scores, which cannot represent valid failure probabilities.

**Conclusion:** Linear Regression was unsuitable for detecting failures at the tested threshold. Logistic Regression was therefore investigated as a method designed for binary classification.